# residuos · dónde falla el modelo de PSF, no cuánto

**Objeto:** ROXs42Bb  |  **Run:** `ROXs42Bb_realigned`  |  **Spec:** [`docs/spec_C1_codex_chromatic_psf.md`](../../../docs/spec_C1_codex_chromatic_psf.md)

El modelo de PSF de C1 reproduce **~28 %** del cromatismo del halo que se mide en el dato (`apcorr_debug` §12), y la §13 de `C1_chromatic_psf_debug` ya demostró que **re-parametrizar lo que hay no lo arregla**. Pero todo eso son cantidades **integradas**: un número por λ, que no dice *dónde* falla.

El residuo `dato − modelo` sí lo dice, y su lectura está cantada de antemano:

| lo que se vea en el residuo | lo que significa |
|---|---|
| anillo a un radio fijo | estructura de la AO: la forma está mal, no la amplitud |
| falda suave que crece al azul | el halo que falta, y su perfil dice qué término añadir |
| **plano en todo el campo** | **pedestal aditivo: es problema del dato, no del modelo** |
| estructura azimutal | elongación, refracción diferencial o vibración |

**Se empieza por la tercera fila** (§5): si el residuo es un pedestal, la medida de los 8 % está inflada y todo lo demás sobra. Las otras tres lecturas son la etapa siguiente de este mismo notebook.

> **No ajusta nada.** El modelo por bin se reconstruye desde el CSV de C1 —`amp · PSF(x, dx, dy) + bck`, con los siete parámetros, el `samp` y el desplazamiento que la etapa ya escribió—, así que esto no vuelve a llamar al optimizador ni una vez.


In [ ]:
import csv, json, sys, warnings
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_aqui = Path.cwd()
ROOT = next(p for p in (_aqui, *_aqui.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)


## 1 · Perillas

Del **config resuelto de la etapa**, nunca copiadas como literales: C1 rellena defaults que el run no escribe, y copiarlos a mano es lo que hizo que el primer notebook de C3 no reprodujera la cadena.


In [ ]:
from musepipe.stages.stage_e01_psf import stage_e01_config_from_run
from musepipe.stages.stage_e01_psfao import prepare_psfao_inputs

E01 = stage_e01_config_from_run(RUN_ID, project_root=ROOT)
R_NORM = float(E01.get('psf_norm_radius_px', E01.get('e01_norm_radius_px', 25.0)))
R_GRANDE = 32.0        # apertura grande: casi todo el halo medible
# Anillos de fondo para la §5.b. NO es una decisión científica nueva:
# se barren tres y se enseña la tendencia, que es el diagnóstico.
ANILLOS = [(40.0, 60.0), (50.0, 70.0), (60.0, 80.0)]
# Los tres tramos en λ con los que se parte todo lo demás.
CORTE_AZUL_A, CORTE_ROJO_A = 6400.0, 8000.0

QC_C1 = json.loads((SD / 'stage_e01_qc.json').read_text(encoding='utf-8'))
PSF_MODEL = json.loads((SD / 'psf_model.json').read_text(encoding='utf-8'))
FORMA = str(PSF_MODEL.get('form', 'moffat')).lower()
ES_PSFAO = FORMA == 'psfao'
NOMBRES = tuple(PSF_MODEL.get('param_names', ()))
print(f'forma elegida por C1: {FORMA}')
print(f'radios: normalización {R_NORM:.0f} px · grande {R_GRANDE:.0f} px')
print(f'anillos de fondo: {ANILLOS}')
_hib = QC_C1.get('hybrid') or {}
print(f"híbrido: aplicado={_hib.get('applied')}"
      f" · suavizado={_hib.get('smoothing_scale_px')}")


## 2 · La escalera de entradas, con su fecha

Un run puede tener piezas de vintages distintos. Si algo aquí está fechado antes que su entrada, lo que salga describe un estado que ya no existe.


In [ ]:
import datetime as _dt

ESCALERA = [
    ('B1/B2  cubo de entrada', SD / 'stage02_xcorr_cube_stack.fits'),
    ('B3     posiciones', SD / 'stage01c_qc.json'),
    ('C1     modelo de PSF', SD / 'psf_model.json'),
    ('C1     parámetros por bin', SD / 'stage_e01_psfao_params.csv'),
    ('C1     residuo del híbrido', SD / 'psf_hybrid_residual.fits'),
]
for etiqueta, ruta in ESCALERA:
    if ruta.exists():
        cuando = _dt.datetime.fromtimestamp(ruta.stat().st_mtime)
        print(f'  {etiqueta:28s} {cuando:%Y-%m-%d %H:%M}  {ruta.name}')
    else:
        print(f'  {etiqueta:28s} {"AUSENTE":16s}  {ruta.name}')


## 3 · Las funciones copiadas de `musepipe`

Viaja copiado lo que este notebook **audita** —el perfil radial del residuo y las métricas de anillo— y se **importa** lo que es de otra etapa y aquí se da por bueno: `evaluate_psf_model` es de C1 y `aperture_correction_from_psf` de C2.

- `source_mask` — de `musepipe/psf.py`
- `companion_ring_metric` — de `musepipe/psf.py`
- `radial_hybrid_profile` — de `musepipe/psf.py`
- `evaluate_radial_profile` — de `musepipe/psf.py`
- `_bad_windows` — de `musepipe/stages/stage_e01_psfao.py`
- `make_bins` — de `musepipe/stages/stage_e01_psfao.py`
- `_ring_residual` — de `musepipe/stages/stage_e01_psfao.py`
- `_as_cube` — de `musepipe/extraction/aperture.py`
- `annulus_background_spectrum` — de `musepipe/extraction/aperture.py`
- `VerificationError` — de `musepipe/reduction/verify.py`
- `circular_aperture_mask` — de `musepipe/reduction/verify.py`
- `extract_aperture_spectrum` — de `musepipe/reduction/verify.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from musepipe.stats import finite_percentile
from scipy.ndimage import gaussian_filter1d
import math
import numpy as np




def source_mask(shape, centers_yx, radius_px):
    yy, xx = np.indices(shape, dtype=np.float64)
    mask = np.zeros(shape, dtype=bool)
    for center in centers_yx or ():
        if center is None:
            continue
        y, x = map(float, center)
        mask |= (yy - y) ** 2 + (xx - x) ** 2 <= float(radius_px) ** 2
    return mask


def companion_ring_metric(image, model, primary_yx, companion_yx, *, width_px=3.0, source_exclusion_radius_px=0.0):
    img = np.asarray(image, dtype=np.float64)
    mod = np.asarray(model, dtype=np.float64)
    yy, xx = np.indices(img.shape, dtype=np.float64)
    py, px = map(float, primary_yx)
    cy, cx = map(float, companion_yx)
    radius = math.hypot(cy - py, cx - px)
    rr = np.sqrt((yy - py) ** 2 + (xx - px) ** 2)
    ann = np.abs(rr - radius) <= float(width_px) / 2.0
    if source_exclusion_radius_px and source_exclusion_radius_px > 0:
        ann &= (yy - cy) ** 2 + (xx - cx) ** 2 > float(source_exclusion_radius_px) ** 2
    halo = np.abs(mod)
    vals = np.abs(img - mod) / np.maximum(halo, np.nanmedian(halo[ann]) * 0.05)
    vals = vals[ann & np.isfinite(vals)]
    if vals.size == 0:
        return {"radius_px": float(radius), "median_pct": np.nan, "p90_pct": np.nan}
    return {
        "radius_px": float(radius),
        "median_pct": float(100.0 * np.nanmedian(vals)),
        "p90_pct": float(100.0 * finite_percentile(vals, 90.0)),
    }


def radial_hybrid_profile(residual, center_yx, *, mask=None, bin_width_px=1.0, smoothing_scale_px=4.0):
    resid = np.asarray(residual, dtype=np.float64)
    yy, xx = np.indices(resid.shape, dtype=np.float64)
    rr = np.sqrt((yy - float(center_yx[0])) ** 2 + (xx - float(center_yx[1])) ** 2)
    valid = np.isfinite(resid)
    if mask is not None:
        valid &= ~np.asarray(mask, dtype=bool)
    bins = np.floor(rr / float(bin_width_px)).astype(int)
    nbin = int(np.nanmax(bins)) + 1
    profile = np.full(nbin, np.nan, dtype=np.float64)
    radii = (np.arange(nbin, dtype=np.float64) + 0.5) * float(bin_width_px)
    for b in range(nbin):
        pix = valid & (bins == b)
        if np.count_nonzero(pix) >= 3:
            profile[b] = np.nanmedian(resid[pix])
    finite = np.isfinite(profile)
    if np.count_nonzero(finite) >= 2:
        profile[~finite] = np.interp(radii[~finite], radii[finite], profile[finite])
    else:
        profile[~finite] = 0.0
    sigma_bins = max(float(smoothing_scale_px) / float(bin_width_px), 0.0)
    if sigma_bins > 0:
        profile = gaussian_filter1d(profile, sigma=sigma_bins, mode="nearest")
    return radii, profile


def evaluate_radial_profile(shape, center_yx, radii, profile):
    yy, xx = np.indices(shape, dtype=np.float64)
    rr = np.sqrt((yy - float(center_yx[0])) ** 2 + (xx - float(center_yx[1])) ** 2)
    return np.interp(rr.ravel(), np.asarray(radii), np.asarray(profile), left=profile[0], right=profile[-1]).reshape(shape)


def _bad_windows(cfg):
    if cfg.get("drop_wave_min_A") is not None and cfg.get("drop_wave_max_A") is not None:
        return [[float(cfg["drop_wave_min_A"]), float(cfg["drop_wave_max_A"])]]
    return [[5780.0, 6050.0]]


def make_bins(wave, bin_A, bad_windows, min_channels=3):
    lo, hi = float(wave.min()), float(wave.max())
    edges = np.arange(lo, hi + bin_A, bin_A)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mid = 0.5 * (a + b)
        if any(w0 <= mid <= w1 for w0, w1 in bad_windows):
            continue
        sel = (wave >= a) & (wave < b)
        if sel.sum() >= min_channels:
            bins.append((a, b, mid, sel))
    return bins


def _ring_residual(image, model, companion_yx, mask_radius, width=1.5):
    ny, nx = image.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - cy, xx - cx)
    comp_r = float(np.hypot(companion_yx[0] - cy, companion_yx[1] - cx))
    comp = np.hypot(yy - companion_yx[0], xx - companion_yx[1])
    ann = (np.abs(r - comp_r) < width) & np.isfinite(image) & (comp > mask_radius)
    halo = np.nanmedian(image[ann])
    if not np.isfinite(halo) or halo == 0:
        return float("nan")
    return float(100.0 * np.nanmedian(np.abs(image[ann] - model[ann])) / halo)


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def annulus_background_spectrum(cube_zyx, center_yx, r_in, r_out, *, exclude_yx=None, exclude_radius=0.0):
    """Per-channel local background = median of a source-free annulus.

    Used for the wings-intact aperture-correction path: subtracting a distant
    annulus (rather than a local surface, stage04b) preserves the companion's
    PSF wings so the PSF growth-curve aperture correction stays self-consistent
    (box3<box5). Excludes a region around ``exclude_yx`` (the primary)."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - float(center_yx[0]), xx - float(center_yx[1]))
    mask = (r >= float(r_in)) & (r <= float(r_out))
    if exclude_yx is not None and float(exclude_radius) > 0:
        mask &= np.hypot(yy - float(exclude_yx[0]), xx - float(exclude_yx[1])) > float(exclude_radius)
    if not mask.any():
        return np.zeros(cube.shape[0], dtype=np.float64)
    vals = cube[:, mask]
    with np.errstate(all="ignore"):
        return np.nanmedian(vals, axis=1).astype(np.float64)


class VerificationError(RuntimeError):
    """Raised when a requested verification cannot be completed."""


def circular_aperture_mask(shape: tuple[int, int], yx: tuple[float, float], radius: float) -> np.ndarray:
    y, x = np.indices(shape, dtype=np.float64)
    cy, cx = yx
    return (y - cy) ** 2 + (x - cx) ** 2 <= radius**2


def extract_aperture_spectrum(cube: np.ndarray, yx: tuple[float, float], radius: float) -> np.ndarray:
    mask = circular_aperture_mask(cube.shape[1:], yx, radius)
    if not mask.any():
        raise VerificationError("Aperture contains no pixels.")
    return np.nansum(cube[:, mask], axis=1)


## 4 · Chequeo de deriva


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/psf.py:source_mask": "21b7a974898a",
    "musepipe/psf.py:companion_ring_metric": "5b37f4cb618a",
    "musepipe/psf.py:radial_hybrid_profile": "24c26c3330fd",
    "musepipe/psf.py:evaluate_radial_profile": "2c72e505513d",
    "musepipe/stages/stage_e01_psfao.py:_bad_windows": "5a64b57438d1",
    "musepipe/stages/stage_e01_psfao.py:make_bins": "26d30a7ccd07",
    "musepipe/stages/stage_e01_psfao.py:_ring_residual": "8392c32f6e2f",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:annulus_background_spectrum": "68ec4c4e7299",
    "musepipe/reduction/verify.py:VerificationError": "d9acb4457343",
    "musepipe/reduction/verify.py:circular_aperture_mask": "b08d990cd3a2",
    "musepipe/reduction/verify.py:extract_aperture_spectrum": "e46a9616dd50"
}

def _pieza(cuerpo, name):
    """El nodo que define `name`: def/class, o la asignación de una constante.

    Las constantes también se vigilan: viajan copiadas igual que las
    funciones, y hasta ahora nadie comprobaba que siguieran siendo las de
    `musepipe` — añadir una banda a un diccionario dejaba esta copia atrás
    sin que nada lo dijera.
    """
    for n in cuerpo:
        if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name:
            inicio = min([n.lineno] + [d.lineno for d in n.decorator_list])
            return inicio, n.end_lineno
        if isinstance(n, _ast.Assign) and any(
                isinstance(t, _ast.Name) and t.id == name for t in n.targets):
            return n.lineno, n.end_lineno
        if (isinstance(n, _ast.AnnAssign) and isinstance(n.target, _ast.Name)
                and n.target.id == name):
            return n.lineno, n.end_lineno
    return None

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        sitio = _pieza(_ast.parse(text).body, name)
        if sitio is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        inicio, fin = sitio
        src = ''.join(lines[inicio - 1:fin]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} RESID')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · ¿Es un pedestal?

La medida que abrió todo esto —`F(≤32 px)/F(≤25 px)`, que varía un **8 %** en el dato y un 2.3 % en el modelo— se hace con dos sumas de apertura **sin restar fondo**. Cualquier componente **aditiva** con dependencia en λ (cielo residual, luz dispersada, pedestal instrumental) entra en el anillo de 25–32 px mucho más que en el círculo interior, y se disfrazaría exactamente de esto.

Cuatro medidas, de la más barata a la más cara. La pregunta es siempre la misma: **¿cuánto del 8 % sobrevive?**


In [ ]:
with fits.open(SD / 'stage02_xcorr_cube_stack.fits', memmap=True) as _h:
    CUBE = np.asarray(_h['CUBES'].data, dtype=float)
    WAVE = np.asarray(_h['WAVELENGTH'].data, dtype=float)
if CUBE.ndim == 4:
    CUBE = CUBE[0]
_pos = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
STAR_YX = tuple(float(v) for v in _pos['primary']['pos_yx'])
COMP_YX = tuple(float(v) for v in _pos['companion']['pos_yx'])
INP = prepare_psfao_inputs(dict(E01), SD)
MASK_RADIUS_PX = float(INP['mask_radius'])
BINS = INP['bins']


def _rango(v):
    """Recorrido de punta a punta, p98/p2 — el mismo de `apcorr_debug` §12."""
    _f = np.isfinite(v)
    return float(np.nanpercentile(v[_f], 98)) / float(np.nanpercentile(v[_f], 2))


F_NORM = extract_aperture_spectrum(CUBE, STAR_YX, R_NORM)
F_GRANDE = extract_aperture_spectrum(CUBE, STAR_YX, R_GRANDE)
with np.errstate(invalid='ignore', divide='ignore'):
    CROMA_DATO = _rango(F_GRANDE / F_NORM)
print(f'cubo {CUBE.shape} · estrella en {STAR_YX} · compañero en {COMP_YX}')
print(f'cromatismo del dato, sin tocar nada: {CROMA_DATO:.4f}'
      f'  ({100 * (CROMA_DATO - 1):.1f} %)   <- el número a explicar')


### 5.a · Lo que el ajuste ya se comió

`fit_bin` ajusta un **fondo constante** `bck` junto con la amplitud, así que un pedestal plano ya debería estar absorbido ahí. Dos preguntas: cuánto pesa ese fondo en el anillo que decide el cociente, y **si es cromático** — porque un pedestal que sube hacia el rojo empujaría el cociente en sentido contrario al que se observa, y entonces explicaría el problema aún menos.


In [ ]:
if not ES_PSFAO:
    print('esta cadena eligió Moffat: su CSV no trae `bck` por bin.')
    print('Las §§5.a, 5.c y 5.d son de la rama psfao; la §5.b sí aplica.')
    FILAS = []
else:
    FILAS = sorted((r for r in csv.DictReader(
        open(SD / 'stage_e01_psfao_params.csv', encoding='utf-8'))
        if r['status'] == 'ok'), key=lambda r: float(r['lambda_A']))
    LAM = np.array([float(r['lambda_A']) for r in FILAS])
    BCK = np.array([float(r['bck']) for r in FILAS])
    _area_anillo = np.pi * (R_GRANDE ** 2 - R_NORM ** 2)
    _flujo_anillo = np.interp(LAM, WAVE, F_GRANDE - F_NORM)
    _peso = BCK * _area_anillo / _flujo_anillo
    _pend = np.polyfit(LAM, BCK, 1)[0] * 1000
    print(f'bck: mediana {np.median(BCK):.4g}, recorrido'
          f' {BCK.min():.3g}..{BCK.max():.3g}')
    print(f'     pesa {np.median(_peso):.2%} del flujo del anillo {R_NORM:.0f}-{R_GRANDE:.0f} px')
    print(f'     pendiente en λ: {_pend:+.3f} por 1000 Å')
    if _pend > 0:
        print('     -> SUBE hacia el rojo. Un aditivo así mete más luz en el anillo')
        print('        en el rojo, o sea empuja F(32)/F(25) HACIA ARRIBA al rojo,')
        print('        y lo que se observa es que BAJA. Explica el problema aún menos.')


### 5.b · La vara, restando fondo

Si el 8 % fuera aditivo, restar un fondo de anillo lo tumbaría. El detalle que hay que mirar **no es sólo el número final, sino la tendencia con el radio del anillo**: cuanto más cerca esté el anillo de la estrella, más halo suyo contiene, y restarlo quita señal de verdad. Si al alejar el anillo el cociente vuelve al valor crudo, lo que se estaba restando era halo, no fondo.


In [ ]:
_n_norm = np.pi * R_NORM ** 2
_n_grande = np.pi * R_GRANDE ** 2
print(f'{"tratamiento":34s} {"cociente":>9s} {"%":>7s} {"nivel restado":>14s}')
print(f'{"sin restar nada":34s} {CROMA_DATO:9.4f}'
      f' {100 * (CROMA_DATO - 1):6.1f}% {"-":>14s}')
SUPERVIVENCIA = []
for _r_in, _r_out in ANILLOS:
    _bkg = annulus_background_spectrum(CUBE, STAR_YX, _r_in, _r_out)
    with np.errstate(invalid='ignore', divide='ignore'):
        _c = _rango((F_GRANDE - _bkg * _n_grande) / (F_NORM - _bkg * _n_norm))
    SUPERVIVENCIA.append((_r_in, _r_out, _c, float(np.nanmedian(_bkg))))
    print(f'{f"anillo {_r_in:.0f}-{_r_out:.0f} px restado":34s} {_c:9.4f}'
          f' {100 * (_c - 1):6.1f}% {np.nanmedian(_bkg):14.4g}')
_cs = np.array([s[2] for s in SUPERVIVENCIA])
_niv = np.array([s[3] for s in SUPERVIVENCIA])
print()
print(f'sobrevive entre {100 * (_cs.min() - 1) / (CROMA_DATO - 1):.0f}%'
      f' y {100 * (_cs.max() - 1) / (CROMA_DATO - 1):.0f}% del cromatismo original.')
if _niv[0] > _niv[-1] and _cs[0] < _cs[-1]:
    print('Y el «fondo» CAE al alejar el anillo mientras el cociente SUBE hacia el')
    print('valor crudo: lo que se restaba era halo de la estrella, no fondo.')


### 5.c · El residuo, región por región

Aquí entra el modelo. Se reconstruye el de cada bin desde el CSV —sin reajustar— y se mira **qué fracción del dato queda sin explicar**, por separado dentro del núcleo (`r ≤ R_NORM`) y en el anillo que decide el cociente. Partido en tres tramos de λ.

Un pedestal aditivo dejaría un residuo del **mismo signo y parecido tamaño en los tres tramos**. Que el residuo del anillo cambie de signo entre el azul y el rojo sería otra cosa: el halo del modelo mal repartido en λ, que es justo lo que la §12 de `apcorr_debug` mide de forma integrada.


In [ ]:
if not ES_PSFAO:
    print('sin CSV de psfao: esta sección no aplica a la forma Moffat.')
else:
    from maoppy.psfmodel import Psfao as _Psfao

    _por_lambda = {round(float(b[2]), 3): b for b in BINS}
    IMAGENES, MODELOS, LAM_OK = [], [], []
    for _r in FILAS:
        _b = _por_lambda.get(round(float(_r['lambda_A']), 3))
        if _b is None:
            continue
        _img = np.nanmedian(CUBE[_b[3]], axis=0)
        _m = _Psfao(_img.shape, system=INP['system'], samp=float(_r['samp']))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            _recon = (float(_r['amp'])
                      * _m([float(_r[n]) for n in NOMBRES],
                           dx=float(_r['dx']), dy=float(_r['dy']))
                      + float(_r['bck']))
        IMAGENES.append(_img); MODELOS.append(_recon)
        LAM_OK.append(float(_r['lambda_A']))
    LAM_OK = np.array(LAM_OK)
    print(f'{len(MODELOS)} modelos por bin reconstruidos desde el CSV, sin reajustar')

    _yy, _xx = np.mgrid[0:IMAGENES[0].shape[0], 0:IMAGENES[0].shape[1]]
    _rr = np.hypot(_yy - STAR_YX[0], _xx - STAR_YX[1])
    MASCARA_COMP = source_mask(IMAGENES[0].shape, [COMP_YX], MASK_RADIUS_PX)
    _sin = ~MASCARA_COMP
    _sel_nucleo = (_rr <= R_NORM) & _sin
    _sel_anillo = (_rr > R_NORM) & (_rr <= R_GRANDE) & _sin
    _S = lambda _im, _sel: np.array([float(np.nansum(_i[_sel])) for _i in _im])
    _d_n, _d_a = _S(IMAGENES, _sel_nucleo), _S(IMAGENES, _sel_anillo)
    _m_n, _m_a = _S(MODELOS, _sel_nucleo), _S(MODELOS, _sel_anillo)
    AZUL = LAM_OK < CORTE_AZUL_A
    ROJO = LAM_OK > CORTE_ROJO_A
    TRAMOS = ((f'azul  <{CORTE_AZUL_A:.0f}', AZUL),
              ('medio', ~AZUL & ~ROJO),
              (f'rojo  >{CORTE_ROJO_A:.0f}', ROJO))
    print()
    print('residuo (dato − modelo) en fracción del dato de cada región:')
    print(f'   {"tramo":16s} {f"r<={R_NORM:.0f}":>10s}'
          f' {f"anillo {R_NORM:.0f}-{R_GRANDE:.0f}":>16s}')
    _fa = {}
    for _et, _sel in TRAMOS:
        _rn = float(np.median(((_d_n - _m_n) / _d_n)[_sel]))
        _ra = float(np.median(((_d_a - _m_a) / _d_a)[_sel]))
        _fa[_et] = _ra
        print(f'   {_et:16s} {_rn:9.2%} {_ra:15.2%}')
    _v = list(_fa.values())
    print()
    if _v[0] * _v[-1] < 0:
        print(f'El residuo del anillo CAMBIA DE SIGNO: {_v[0]:+.1%} en el azul,'
              f' {_v[-1]:+.1%} en el rojo.')
        print('El modelo se queda corto de halo en el azul y se pasa en el rojo.')
        print('Eso NO es un pedestal aditivo: es el halo mal repartido en λ.')
    else:
        print('El residuo del anillo mantiene el signo en toda la banda:')
        print('compatible con una componente aditiva — seguir por la §5.d.')


### 5.d · El perfil radial del residuo

La prueba directa, y la que no admite interpretación: **un pedestal es plano en radio**. Se toma la mediana azimutal del residuo por bin —la misma función que usa la rama híbrida de C1— y se normaliza a `R_NORM`, por tramos de λ.

Plano de 20 a 60 px → aditivo. Cayendo → es la falda de la estrella, y entonces el 8 % es real y el problema está en el modelo.


In [ ]:
if not ES_PSFAO:
    print('sin CSV de psfao: esta sección no aplica a la forma Moffat.')
else:
    SUAVE_PX = float((QC_C1.get('hybrid') or {}).get('smoothing_scale_px', 0.0))
    PERFILES, RADIOS = [], None
    for _img, _mod in zip(IMAGENES, MODELOS):
        _rad, _per = radial_hybrid_profile(_img - _mod, STAR_YX,
                                          mask=MASCARA_COMP,
                                          smoothing_scale_px=SUAVE_PX)
        if RADIOS is None:
            RADIOS = _rad
        PERFILES.append(_per if _rad.size == RADIOS.size
                        else np.interp(RADIOS, _rad, _per))
    PERFILES = np.array(PERFILES)
    _i_norm = int(np.argmin(np.abs(RADIOS - R_NORM)))
    _muestra = (20.0, R_NORM, R_GRANDE, 40.0, 50.0, 60.0)
    print(f'perfil radial del residuo, normalizado a r={R_NORM:.0f}:')
    print(f'   {"tramo":16s} ' + '  '.join(f'r={_r:<5.0f}' for _r in _muestra))
    _curvas = []
    for _et, _sel in TRAMOS:
        _p = np.nanmedian(PERFILES[_sel], axis=0)
        _p = _p / _p[_i_norm]
        _curvas.append((_et, _p))
        print(f'   {_et:16s} ' + '  '.join(
            f'{_p[int(np.argmin(np.abs(RADIOS - _r)))]:6.2f} ' for _r in _muestra))
    _caida = [np.abs(_p[int(np.argmin(np.abs(RADIOS - 60.0)))]) for _, _p in _curvas]
    print()
    if max(_caida) < 0.3:
        print('A 60 px no queda nada del residuo: CAE como una falda, no es plano.')
        print('-> la hipótesis del pedestal aditivo queda descartada.')
    else:
        print('El residuo sigue en pie a 60 px: compatible con un pedestal.')
    fig, ax = plt.subplots(figsize=(9, 3.6))
    for _et, _p in _curvas:
        ax.plot(RADIOS, _p, lw=1.1, label=_et)
    ax.axvline(R_NORM, color='0.6', lw=0.7, ls=':')
    ax.axvline(R_GRANDE, color='0.6', lw=0.7, ls=':')
    ax.axhline(0.0, color='0.6', lw=0.7)
    ax.set_xlim(0, 70); ax.set_xlabel('radio [px]')
    ax.set_ylabel(f'residuo / residuo(r={R_NORM:.0f})', fontsize=8)
    ax.set_title('§5.d · plano = pedestal; cayendo = falda de la estrella',
                 fontsize=9)
    ax.legend(fontsize=8); fig.tight_layout(); plt.show()


## 6 · El término híbrido que C1 mide, guarda y no hace viajar

C1 ya calcula este residuo. La spec (§3.5) le llama **híbrido**: si el residuo del anillo del compañero supera el 5 % en más del 20 % de los bins, se le añade al modelo la **mediana azimutal del residuo**, suavizada, que es azimutalmente simétrica y por eso no puede absorber una fuente puntual. En este run está **aplicado**, y su producto es el `psf_hybrid_residual.fits` que la §5.d ya usó.

Pero el término **no entra en `psf_model.json`**: el documento lleva una bandera `hybrid: true` que `_evaluate_psfao` no mira (§11 de `C1_chromatic_psf_debug`). O sea que C2–C6 evalúan un modelo **sin** él.

La pregunta obvia era: **¿es éste el término que falta?** Si lo fuera, el problema del halo estaría medido y guardado desde el principio, y sólo faltaría hacerlo viajar. La sección lo mide en la región que importa.


In [ ]:
# El termino, tal y como la cadena lo guarda. Se carga aqui porque esta
# seccion es la primera que lo usa; la §9 lo compara contra la copia.
_ruta_hib = SD / 'psf_hybrid_residual.fits'
_prod = _prod_r = None
if ES_PSFAO and _ruta_hib.exists():
    with fits.open(_ruta_hib) as _h:
        _prod = np.asarray(_h['PROFILE'].data, dtype=np.float64)
        _prod_r = np.asarray(_h['RADIUS_PX'].data, dtype=np.float64)

if _prod is None:
    print('sin término híbrido en disco: esta sección no aplica.')
else:
    HIBRIDOS = [_m + evaluate_radial_profile(_m.shape, STAR_YX, _prod_r, _p)
                for _m, _p in zip(MODELOS, _prod)]
    _h_a = _S(HIBRIDOS, _sel_anillo)
    print('residuo del anillo'
          f' {R_NORM:.0f}-{R_GRANDE:.0f} px, en fracción del dato:')
    print(f'   {"tramo":16s} {"sin híbrido":>13s} {"con híbrido":>13s}')
    _empeora = 0
    for _et, _sel in TRAMOS:
        _sin = float(np.median(((_d_a - _m_a) / _d_a)[_sel]))
        _con = float(np.median(((_d_a - _h_a) / _d_a)[_sel]))
        _empeora += int(abs(_con) > abs(_sin))
        print(f'   {_et:16s} {_sin:12.2%} {_con:13.2%}')
    print()
    # Y lo que el propio CSV publica: la metrica por la que la cadena
    # JUZGA al hibrido, que es el anillo del COMPAÑERO, no este.
    _antes = np.array([float(r['ring_residual_pct_canonical']) for r in FILAS
                       if r.get('ring_residual_pct_canonical')])
    _desp = np.array([float(r['ring_residual_pct_after_hybrid_canonical'])
                      for r in FILAS
                      if r.get('ring_residual_pct_after_hybrid_canonical')])
    if _antes.size and _desp.size:
        print(f'la métrica por la que la cadena lo juzga —el anillo del COMPAÑERO,'
              f' a ~{np.hypot(COMP_YX[0] - STAR_YX[0], COMP_YX[1] - STAR_YX[1]):.0f} px—'
              f' mejora: {np.median(_antes):.2f} % → {np.median(_desp):.2f} %')
    if _empeora >= 2:
        print()
        print('Pero en el anillo que fija la corrección de apertura EMPEORA.')
        print('El término mejora donde se le mide y estropea donde nadie mira.')


### 6.a · Por qué empeora: el suavizado arrastra el núcleo

`radial_hybrid_profile` toma la mediana azimutal por anillos de 1 px y luego la **suaviza con una gaussiana de σ ≥ 2×FWHM**. Esa escala es una salvaguarda deliberada de la spec —garantiza que el término no pueda absorber al compañero—, pero tiene un precio: a `R_NORM` la ventana toca radios muy interiores, y el residuo del **núcleo** es mucho mayor que el del halo.

La celda compara el perfil tal cual sale (sin suavizar) con el que la cadena guarda. Si el suavizado infla el término varias veces en 25–32 px, el híbrido que hay en disco no es «el residuo del halo»: es el residuo del núcleo, esparcido.


In [ ]:
if _prod is not None:
    CRUDOS = []
    for _img, _mod in zip(IMAGENES, MODELOS):
        _r0, _p0 = radial_hybrid_profile(_img - _mod, STAR_YX,
                                        mask=MASCARA_COMP, smoothing_scale_px=0.0)
        CRUDOS.append(_p0 if _r0.size == RADIOS.size
                      else np.interp(RADIOS, _r0, _p0))
    CRUDOS = np.array(CRUDOS)
    print(f'suavizado del término: σ = {SUAVE_PX:.1f} px'
          f'  (la spec §3.5 pide ≥ 2×FWHM)')
    print()
    print(f'   {"tramo":10s} {"r":>4s} {"sin suavizar":>13s} {"guardado":>11s}'
          f' {"factor":>8s}')
    _factores = []
    for _et, _sel in TRAMOS:
        for _r in (R_NORM, R_GRANDE):
            _i = int(np.argmin(np.abs(RADIOS - _r)))
            _a = float(np.median(CRUDOS[_sel, _i]))
            _b = float(np.median(PERFILES[_sel, _i]))
            _f = _b / _a if _a else np.nan
            _factores.append(abs(_f))
            print(f'   {_et:10s} {_r:4.0f} {_a:13.3g} {_b:11.3g} {_f:8.2f}')
    _i5 = int(np.argmin(np.abs(RADIOS - 5.0)))
    _i25 = int(np.argmin(np.abs(RADIOS - R_NORM)))
    print()
    print(f'residuo sin suavizar en el núcleo (r=5 px):'
          f' {np.median(np.abs(CRUDOS[:, _i5])):.4g}')
    print(f'   ... y en r={R_NORM:.0f} px:'
          f' {np.median(np.abs(CRUDOS[:, _i25])):.4g}'
          f'   (×{np.median(np.abs(CRUDOS[:, _i5])) / max(np.median(np.abs(CRUDOS[:, _i25])), 1e-30):.1f}'
          ' más grande dentro)')
    if np.nanmax(_factores) > 3:
        print()
        print(f'El suavizado infla el término hasta ×{np.nanmax(_factores):.0f} en la'
              ' zona de la apcorr, y llega a cambiarle el signo.')
        print('-> el híbrido guardado NO es el término que le falta al halo:')
        print('   es el residuo del núcleo, esparcido hacia fuera.')


## 7 · El mapa (radio, λ)

La figura que resume todo: el residuo azimutal-mediano, **normalizado al perfil de la propia estrella** en cada bin para que los colores comparen forma y no brillo. Lo que se busca es estructura:

- una **franja vertical** a un radio fijo → anillo de la AO (o el radio de corte del sistema), y entonces el modelo tiene la forma mal;
- un **degradado en λ** al radio de la apcorr → el halo mal repartido, que es lo que la §5.c ya midió con números.


In [ ]:
if ES_PSFAO and _prod is not None:
    _perf_estrella = np.array([
        radial_hybrid_profile(_img, STAR_YX, mask=MASCARA_COMP,
                              smoothing_scale_px=0.0)[1]
        for _img in IMAGENES])
    with np.errstate(invalid='ignore', divide='ignore'):
        _mapa = CRUDOS / np.abs(_perf_estrella)
    _hasta = int(np.argmin(np.abs(RADIOS - 70.0)))
    _v = np.nanpercentile(np.abs(_mapa[:, :_hasta]), 98)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    _im = ax.pcolormesh(RADIOS[:_hasta], LAM_OK, _mapa[:, :_hasta],
                        cmap='RdBu_r', vmin=-_v, vmax=_v, shading='nearest')
    for _r, _et in ((R_NORM, 'R_NORM'), (R_GRANDE, 'R_GRANDE'),
                    (np.hypot(COMP_YX[0] - STAR_YX[0],
                              COMP_YX[1] - STAR_YX[1]), 'compañero')):
        if _r < 70:
            ax.axvline(_r, color='k', lw=0.8, ls=':')
            ax.text(_r, LAM_OK.max(), f' {_et}', fontsize=7, va='top')
    ax.set_xlabel('radio [px]'); ax.set_ylabel('λ [Å]')
    ax.set_title('§7 · residuo / perfil de la estrella  '
                 '(rojo = al modelo le falta luz; azul = le sobra)', fontsize=9)
    fig.colorbar(_im, ax=ax, label='fracción del perfil')
    fig.tight_layout(); plt.show()
    _i_ap = slice(int(np.argmin(np.abs(RADIOS - R_NORM))),
                  int(np.argmin(np.abs(RADIOS - R_GRANDE))) + 1)
    _franja = np.nanmedian(_mapa[:, _i_ap], axis=1)
    print(f'en la banda {R_NORM:.0f}-{R_GRANDE:.0f} px, el residuo relativo va de'
          f' {np.median(_franja[AZUL]):+.1%} (azul) a {np.median(_franja[ROJO]):+.1%} (rojo)')
else:
    print('sin CSV de psfao: esta sección no aplica a la forma Moffat.')


## 8 · ¿Tiene estructura azimutal?

Todo lo anterior promedia en ángulo, así que **por construcción no puede ver** una elongación, la refracción diferencial o una vibración: las tres dejan el residuo repartido de forma distinta según la dirección. Aquí se mira el residuo en sectores a los radios que importan.

Si la amplitud azimutal es pequeña frente al residuo medio, el término que falta es **azimutalmente simétrico**, y eso descarta de golpe a la refracción diferencial —que tiene una dirección, la del ángulo paraláctico— como explicación principal.


In [ ]:
if ES_PSFAO:
    _N_SECT = 12
    _ang = np.degrees(np.arctan2(_yy - STAR_YX[0], _xx - STAR_YX[1])) % 360.0
    _sep_comp = float(np.hypot(COMP_YX[0] - STAR_YX[0], COMP_YX[1] - STAR_YX[1]))
    print(f'{"radio":>7s} {"tramo":10s} {"media":>10s} {"amplitud azimutal":>18s}'
          f' {"máx en":>8s}')
    for _r in (R_NORM, R_GRANDE, _sep_comp):
        _corona = (np.abs(_rr - _r) <= 2.0) & (~MASCARA_COMP)
        for _et, _sel in TRAMOS:
            _res = np.nanmedian([_i - _m for _i, _m in
                                 zip(np.array(IMAGENES)[_sel],
                                     np.array(MODELOS)[_sel])], axis=0)
            _val = []
            for _k in range(_N_SECT):
                _w = _corona & (_ang >= _k * 360 / _N_SECT) & (_ang < (_k + 1) * 360 / _N_SECT)
                _val.append(np.nanmedian(_res[_w]) if _w.any() else np.nan)
            _val = np.array(_val)
            _med = float(np.nanmean(_val))
            _amp = float(np.nanmax(_val) - np.nanmin(_val))
            _donde = float((np.nanargmax(_val) + 0.5) * 360 / _N_SECT)
            print(f'{_r:7.0f} {_et:10s} {_med:10.3g} {_amp:18.3g}'
                  f' {_donde:7.0f}°')
    print()
    print('Si la amplitud es del orden de la media, hay dirección privilegiada.')
    print('Si es mucho menor, el término que falta es simétrico, y la refracción')
    print('diferencial —que tiene dirección— no es la explicación principal.')
    # El sospechoso obvio antes de pensar en la atmósfera: la elipse del
    # PROPIO modelo. Un eje mal puesto deja residuo cuadrupolar alineado
    # con él, y eso no es física del cielo, es el ajuste.
    _th = np.array([float(_r['theta']) for _r in FILAS])
    _rt = np.array([float(_r['ratio']) for _r in FILAS])
    _eje = np.degrees(np.where(_rt >= 1, _th, _th + np.pi / 2)) % 180.0
    print()
    print(f'el eje mayor de la elipse del modelo está en'
          f' {np.median(_eje):.0f}° (mod 180)')
    print('   si el máximo azimutal del residuo cae sobre ese mismo eje, lo que')
    print('   sobra es elipticidad mal ajustada, no un fenómeno atmosférico.')
else:
    print('sin CSV de psfao: esta sección no aplica a la forma Moffat.')


## 9 · Comparación con la cadena

El producto que esta reconstrucción tiene que reproducir es **`psf_hybrid_residual.fits`**: la mediana azimutal de `dato − modelo` por bin, que C1 calcula en su rama híbrida. Es una **función pura** de las imágenes por bin, los modelos y la máscara del compañero — y las tres se rehacen aquí desde el CSV, sin reajustar.

Si esto sale idéntico, la reconstrucción del modelo es la de la cadena, y todo lo que dice la §5 se apoya en el mismo modelo que consumen C2–C6.

> El producto se guarda en **float32** y aquí se calcula en float64, así que la tolerancia es la del redondeo de almacenamiento (~1e-5 relativo), no un margen elegido a conveniencia.


In [ ]:
if _prod is None:
    print('no hay `psf_hybrid_residual.fits` para esta forma/run:'
          ' la comparación no aplica.')
else:
    print(f'perfiles: reconstruidos {PERFILES.shape} · producto {_prod.shape}')
    _ok = PERFILES.shape == _prod.shape and np.allclose(RADIOS, _prod_r)
    if _ok:
        _esc = float(np.nanmedian(np.abs(_prod))) or 1.0
        _dif = float(np.nanmax(np.abs(PERFILES - _prod)))
        _tol = 1e-4 * _esc      # redondeo de float32 sobre la escala del perfil
        print(f'   |dif| máx = {_dif:.4g}   escala = {_esc:.4g}'
              f'   relativo = {_dif / _esc:.2e}')
        _ok = _dif <= _tol
    print()
    # La frase es literal a proposito: el test `slow` la busca tal cual,
    # igual que en los otros nueve notebooks.
    print('IDÉNTICO: la copia reproduce la cadena.' if _ok else
          'DIFIERE — si has tocado una perilla, es lo esperado;'
          ' si no, mira el chequeo de deriva.')


## 10 · Qué NO decide este notebook

1. **No dice de qué está hecho lo que sobra.** Separar «aditivo» de «halo» no identifica la causa: cielo residual, luz dispersada en el instrumento, fringing y refracción diferencial dejan firmas distintas, y ninguna se mide aquí.
2. **No toca C1 ni la cadena.** Cambiar el modelo obliga a re-correr C1 → C2–C6 → D2 → E → F → G, y es decisión científica.
3. **El anillo de fondo de la §5.b no es una decisión nueva**: se barren tres radios y lo que se lee es la *tendencia*, no un valor elegido.

4. **No dice que el híbrido esté mal calculado.** La §6 mide que, tal y como se guarda, no sirve para el halo —el suavizado que lo hace seguro frente al compañero es el que lo estropea a 25–32 px—, no que la rama híbrida sea un error. Con otra escala de suavizado sería otra cosa, y eso **no se prueba aquí**.
5. **No propone hacerlo viajar.** Meter un término radial en `psf_model.json` obliga a que `evaluate_psf_model` lo evalúe, y eso mueve todo lo que consume la PSF de C1.

**Lo que queda por hacer, y ya no es de este notebook:** con el residuo del anillo cambiando de signo entre el azul y el rojo, y siendo simétrico en ángulo, el siguiente paso es **pedirle el halo al ajuste** —pesar radialmente el χ², o ajustar contra `F(r, λ)` en vez de contra la imagen— que es la hipótesis 2 del traspaso del 2026-08-13, la única de las tres que sigue en pie sin tocar.
